# Proper imports...

In [1]:
# Set up monitoring of changes to the module
#using Revise
using Pkg
Pkg.activate(".")

# Load module and others
using MPSCircuits, ITensors, ITensorMPS, LinearAlgebra

  Activating project at `~/Documents/mps-state-prep/MPSCircuits`

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
Revise.revise()

In [ ]:
Revise.errors()

# Transpiled gates test

# Compilation test

In [2]:
# 1. Setup Parameters
N = 50
j_coupling = 1.0
h_field = 0.5  # Transverse field

# 2. Define Site Indices
# "S=1/2" is the standard site type for qubits/spins
sites = siteinds("S=1/2", N)

# 3. Construct Hamiltonian using OpSum
os = OpSum()
for j in 1:(N - 1)
  # Interaction term: -J * Z_i * Z_{i+1}
  os += -j_coupling, "Sz", j, "Sz", j + 1
end
for j in 1:N
  # Field term: -h * X_i
  os += -h_field, "Sx", j
end

# Convert OpSum to MPO
H = MPO(os, sites)

# 4. Initialize State
# Start with a random product state (bond dimension 1)
# or a specific one like "Up"
state = [isodd(n) ? "Up" : "Dn" for n in 1:N]
psi_init = MPS(sites, state)

# 5. DMRG Settings (Sweeps)
# Each sweep gradually increases bond dimension (maxdim) 
# and decreases the truncation error (cutoff)
nsweeps = 5
maxdim = [10, 20, 100]
cutoff = [1e-10]

# 6. Run DMRG
energy, mps = dmrg(H, psi_init; nsweeps, maxdim, cutoff)

println("Ground State Energy: ", energy)

After sweep 1 energy=-15.806600464330774  maxlinkdim=4 maxerr=3.32E-16 time=14.958
After sweep 2 energy=-15.824805632828388  maxlinkdim=13 maxerr=9.95E-11 time=0.052
After sweep 3 energy=-15.825234680395129  maxlinkdim=18 maxerr=9.95E-11 time=0.094
After sweep 4 energy=-15.825290952266425  maxlinkdim=17 maxerr=9.88E-11 time=0.080
After sweep 5 energy=-15.825297037672172  maxlinkdim=16 maxerr=9.91E-11 time=0.068
Ground State Energy: -15.825297037672172


In [4]:
circuit = MPSCircuits.compile_mps_circuit(mps, MPSCircuits.DecomposeAllAnalytical(); n_layers_max=10, tolerance=1e-4)
MPSCircuits.evaluate_circuit_fidelity(circuit, mps; cutoff=1e-12)

0.9632837360396372

In [17]:
circuit_transpiled = [MPSCircuits.KAKGateSU4(gate) for gate in circuit]
circuit_expanded = MPSCircuits.expand_SU4s(circuit_transpiled)

2450-element Vector{MPSCircuits.AbstractGate}:
 MPSCircuits.SU2Gate{MPSCircuits.ZYZ}(ITensor ord=2
Dim 1: (dim=2|id=191|"S=1/2,Site,n=1")'
Dim 2: (dim=2|id=191|"S=1/2,Site,n=1")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 0.6533134341291429 - 0.27058481460202355im  …  -0.6532495291844149 - 0.2706112848968327im
 0.6532495291844149 - 0.2706112848968326im       0.6533134341291429 + 0.27058481460202366im
, [1], Index{Int64}[(dim=2|id=191|"S=1/2,Site,n=1")], 0.7853981657897076, 1.5707271565811902, -6.917021387159439e-5)
 MPSCircuits.SU2Gate{MPSCircuits.ZYZ}(ITensor ord=2
Dim 1: (dim=2|id=210|"S=1/2,Site,n=2")'
Dim 2: (dim=2|id=210|"S=1/2,Site,n=2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
  -0.6532617526768998 + 0.2705504216881218im  …  0.27064567701979814 - 0.6533012087271686im
 -0.27064567701979825 - 0.6533012087271687im        -0.6532617526769 - 0.2705504216881217im
, [2], Index{Int64}[(dim=2|id=210|"S=1/2,Site,n=2")], -0.7853981687126275, 1.5708994304228483, -4.712

In [19]:
MPSCircuits.evaluate_circuit_fidelity(circuit_expanded, mps; cutoff=1e-12)

0.9632741503703716

In [18]:
Revise.revise()

In [50]:
fused_circuit = MPSCircuits.fuse_SU2s(circuit_expanded, length(sites))

1520-element Vector{MPSCircuits.AbstractGate}:
 MPSCircuits.SU2Gate{MPSCircuits.ZYZ}(ITensor ord=2
Dim 1: (dim=2|id=191|"S=1/2,Site,n=1")'
Dim 2: (dim=2|id=191|"S=1/2,Site,n=1")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
 0.6533134341291429 - 0.27058481460202355im  …  -0.6532495291844149 - 0.2706112848968327im
 0.6532495291844149 - 0.2706112848968326im       0.6533134341291429 + 0.27058481460202366im
, [1], Index{Int64}[(dim=2|id=191|"S=1/2,Site,n=1")], 0.7853981657897076, 1.5707271565811902, -6.917021387159439e-5)
 MPSCircuits.SU2Gate{MPSCircuits.ZYZ}(ITensor ord=2
Dim 1: (dim=2|id=210|"S=1/2,Site,n=2")'
Dim 2: (dim=2|id=210|"S=1/2,Site,n=2")
NDTensors.Dense{ComplexF64, Vector{ComplexF64}}
 2×2
  -0.6532617526768998 + 0.2705504216881218im  …  0.27064567701979814 - 0.6533012087271686im
 -0.27064567701979825 - 0.6533012087271687im        -0.6532617526769 - 0.2705504216881217im
, [2], Index{Int64}[(dim=2|id=210|"S=1/2,Site,n=2")], -0.7853981687126275, 1.5708994304228483, -4.712

In [43]:
MPSCircuits.evaluate_circuit_fidelity(fused_circuit, mps; cutoff=1e-12)

0.9632741503703554

In [45]:
length(circuit)

490

In [39]:
length(circuit_expanded)

2450

In [38]:
length(fused_circuit)

1470

In [49]:
circuit = MPSCircuits.compile_mps_circuit(mps, MPSCircuits.IterativeDecomposeOptimizeLayer(); n_layers_max=10, n_iterations_per_layer=5, tolerance=1e-4)
MPSCircuits.evaluate_circuit_fidelity(circuit, mps; cutoff=1e-12)

0.9597706083470331

In [ ]:
circuit = MPSCircuits.compile_mps_circuit(mps, MPSCircuits.IterativeDecomposeOptimizeAll(); n_layers_max=10, n_iterations_per_layer=1, tolerance=1e-4)
MPSCircuits.evaluate_circuit_fidelity(circuit, mps; cutoff=1e-12)

0.9983607149430115

In [16]:
circuit = MPSCircuits.compile_mps_circuit(mps, MPSCircuits.IterativeDecomposeOptimizeAll(); optimization_protocol=MPSCircuits.RollingEnvironment(), n_layers_max=10, n_iterations_per_layer=1, tolerance=1e-4)
MPSCircuits.evaluate_circuit_fidelity(circuit, mps; cutoff=1e-12)

0.9982591271901708

In [20]:
circuit = MPSCircuits.compile_mps_circuit(mps, MPSCircuits.IterativeDecomposeOptimizeAll(); optimization_protocol=MPSCircuits.TelescopingEnvironment(), n_layers_max=10, n_iterations_per_layer=1, tolerance=1e-4)
MPSCircuits.evaluate_circuit_fidelity(circuit, mps; cutoff=1e-12)

0.9983607149430115

# Old tests

First generate an MPS to test out...

In [2]:
include("fcidump_to_mpo.jl")

twoterm_labels

In [3]:
mps, energy = fcidump_to_mps("chemistry-examples/h2_631g_rhf.fcidump")

nel=2
norb=4
ns=0
Initial state:[2, 2, 1, 1, 1, 1, 1, 1]
After sweep 1 energy=-1.1267783526183102  maxlinkdim=41 maxerr=3.45E-41 time=13.692
After sweep 2 energy=-1.12677835261831  maxlinkdim=40 maxerr=6.41E-41 time=0.126
After sweep 3 energy=-1.1267783526183097  maxlinkdim=39 maxerr=8.21E-41 time=0.043
After sweep 4 energy=-1.1267783526183102  maxlinkdim=42 maxerr=6.98E-41 time=0.043
After sweep 5 energy=-1.1267783526183102  maxlinkdim=43 maxerr=8.37E-41 time=0.058
After sweep 6 energy=-1.1267783526183097  maxlinkdim=41 maxerr=4.69E-41 time=0.042
After sweep 7 energy=-1.1267783526183102  maxlinkdim=41 maxerr=7.70E-41 time=0.044
After sweep 8 energy=-1.12677835261831  maxlinkdim=40 maxerr=9.58E-41 time=0.040
After sweep 9 energy=-1.1267783526183097  maxlinkdim=42 maxerr=3.88E-41 time=0.043
After sweep 10 energy=-1.1267783526183102  maxlinkdim=41 maxerr=9.00E-41 time=0.043
Final energy = -1.1267783526183102


(MPS(8), -1.1267783526183102)